# Validación de datos experimentales offline

Objetivo: verificar configuración, disponibilidad, integridad, splits y consultas de los cuatro datasets antes de ejecutar inferencia. Cada imagen del dataset de habitaciones representa una observación independiente.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_navigation_ws" / "src").is_dir())
sys.path.insert(0, str(repo / "experiments" / "shared"))

import importlib.metadata as metadata
import pandas as pd

from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from semantic_evaluation.core.config_validation import validate_offline_isolation
from semantic_evaluation.core.dataset_adapters import load_dataset, validate_dataset
from semantic_evaluation.core.offline_dataset import load_queries, validate_queries

ctx = bootstrap_offline()
config = ctx["config"]
validate_offline_isolation(config, str(ctx["config_path"]))
print(f"Configuración: {ctx['config_path']}")
print(f"Dispositivo: {ctx['device']} · semilla: {config['experiment']['seed']}")


## Entorno y modelos


In [ ]:
packages = ["numpy", "pandas", "torch", "transformers", "ultralytics", "Pillow"]
versions = []
for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "no instalado"
    versions.append({"package": package, "version": version})
display(pd.DataFrame(versions))
display(pd.DataFrame([
    {"model": "SigLIP", **config["models"]["siglip"]},
    {"model": "YOLO", **config["models"]["yolo"]},
]))


## Disponibilidad e integridad


In [ ]:
bundles = {}
issue_rows = []
for spec in ctx["dataset_specs"]:
    bundle = load_dataset(spec, ctx["repo_root"])
    bundles[spec.dataset_id] = bundle
    issue_rows.extend({"dataset_id": spec.dataset_id, **issue}
                      for issue in validate_dataset(bundle))

availability = pd.DataFrame([
    {"dataset_id": key, "split": value.split, "available": not value.skipped,
     "nodes": len(value.nodes), "topology_edges": len(value.topology_edges),
     "status": value.skip_reason or "available"}
    for key, value in bundles.items()
])
display(availability)
display(pd.DataFrame(issue_rows))


## Splits y consultas


In [ ]:
query_rows = []
query_issues = []
for spec in ctx["dataset_specs"]:
    bundle = bundles[spec.dataset_id]
    query_path = resolve_repo_path(ctx["repo_root"], spec.queries_file)
    queries = load_queries(str(query_path))
    if not bundle.skipped and bundle.nodes:
        query_issues.extend(
            {"dataset_id": spec.dataset_id, "issue": issue}
            for issue in validate_queries(queries, bundle)
        )
    query_rows.append({"dataset_id": spec.dataset_id, "queries": len(queries),
                       "buildings_by_split": bundle.metadata.get("building_split_by_scan", {})})
display(pd.DataFrame(query_rows))
display(pd.DataFrame(query_issues))


## Plantillas y manifiestos


In [ ]:
import json
from dataclasses import asdict
from reproducibility import collect_manifest, save_manifest
from semantic_evaluation.core.dataset_adapters import matterport_annotation_template

template_root = resolve_repo_path(ctx["repo_root"], config["paths"]["annotation_templates_root"])
manifest_root = resolve_repo_path(ctx["repo_root"], config["paths"]["manifests_root"])
template_root.mkdir(parents=True, exist_ok=True)
manifest_root.mkdir(parents=True, exist_ok=True)

for spec in ctx["dataset_specs"]:
    bundle = bundles[spec.dataset_id]
    if spec.dataset_id == "matterport3d" and not bundle.skipped:
        (template_root / "matterport3d_annotations.yaml").write_text(
            matterport_annotation_template(bundle), encoding="utf-8")
    manifest = collect_manifest(
        config, repo_dir=str(ctx["repo_root"]), device=ctx["device"],
        extra={"notebook": "00_data_validation", "dataset_id": spec.dataset_id,
               "split": bundle.split, "item_ids": bundle.node_ids(),
               "status": "skipped" if bundle.skipped else "validated",
               "skip_reason": bundle.skip_reason},
    )
    save_manifest(str(manifest_root / f"{spec.dataset_id}.json"), manifest)
print(f"Manifiestos: {manifest_root}")


## Interpretación


In [ ]:
available = availability.loc[availability["available"], "dataset_id"].tolist()
missing = availability.loc[~availability["available"], ["dataset_id", "status"]]
print(f"Datasets listos: {available or 'ninguno'}")
if not missing.empty:
    print("Los experimentos asociados a las siguientes fuentes se omitirán:")
    display(missing)
if issue_rows:
    print("Las incidencias anteriores deben resolverse antes de usar el dataset afectado.")
